In [ ]:
from google.colab import drive, userdata
import os

drive_dir = "/content/drive"
path = "MyDrive/Projects & Self Study/Building an LLM from scratch"

full_path = os.path.join(drive_dir, path)
drive.mount(drive_dir, force_remount=True)

os.chdir(full_path)
print(f"Working Directory set to: {os.getcwd()}")

Mounted at /content/drive
Working Directory set to: /content/drive/MyDrive/Projects & Self Study/Building an LLM from scratch


In [ ]:
import torch
import torch.nn as nn
import sys

from chapter_3_attention import MultiHeadAttention

#GPT Model from Scratch

###Dummy Classes

Dictionary configuration that we pass as parameter to the class

In [ ]:
GPT_CONFIG_124M = {
    "emb_dim": 768,
    "num_layers": 12,
    "num_heads": 12,
    "context_length": 1024,
    "vocab_size": 50257,
    "drop_rate": 0.1,
    "qkv_bias": True,
}

In [ ]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        # Placeholder
        super(DummyTransformerBlock, self).__init__()

    def forward(self, x):
        # Does nothing, for now
        return x

In [ ]:
class DummyLayerNorm(nn.Module):
    def __init__(self, cfg):
        super(DummyLayerNorm, self).__init__()

    def forward(self, x):
        # Does nothing, for now
        return x

In [ ]:
class DummyGPTModel(nn.Module):
    # cfg = configuration, python dictionary
    def __init__(self, cfg):
        super(DummyGPTModel, self).__init__()
        # From vocabulary size (50257 from the above cell) to embedding dimension (768)
        # From a 50257 dimensional vector to a 768 dimensional vector
        # From the the vocabulary size to an intermediate representation
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Repeated transformer blocks, by the amount of # of layers
        self.transf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["num_layers"])]
        )

        # Layer normalizations applies optimization steps to improve training
        # After normalization, the layers have zero-centered mean and unit variance
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        # From the embedding space to the vocabulary size
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        # [batch_size, sequence_length]
        # Create embeddings
        batch_size, seq_len = in_idx.shape
        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len))

        # Pass x through all the layers
        # Dropout, transformer blocks, normalizations, take final logits
        # They represent a statistical distribution over the vocabulary
        x = tok_emb + pos_emb
        x = self.drop_emb(x)
        x = self.transf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

batch = []

text_one = "I like cats and"
text_two = "I like pizza and"

batch.append(torch.tensor(tokenizer.encode(text_one)))
batch.append(torch.tensor(tokenizer.encode(text_two)))
# dim=0, stack the two rows
batch = torch.stack(batch, dim=0)

print(batch)

tensor([[   40,   588, 11875,   290],
        [   40,   588, 14256,   290]])


The result is a batch of two tensor that are $4$ tokens long, and each token is represented by a $50257$ dimensional vector.

In [ ]:
torch.manual_seed(123)
model = DummyGPTModel(cfg=GPT_CONFIG_124M)

logits = model(batch)

print(f"Output shape: {logits.shape}")
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.7336, -1.1718,  0.0319,  ..., -0.5045,  0.3875, -0.0966],
         [-1.3008,  0.3405, -1.3143,  ...,  0.1798, -0.6737,  0.6819],
         [-1.5930,  0.8470,  0.2528,  ..., -0.0606,  0.2729, -0.3373],
         [-0.6568,  0.0516,  2.2908,  ...,  0.5664,  1.6118,  0.8661]],

        [[ 0.3228, -0.9859, -0.1417,  ..., -0.2459,  0.8905, -0.0729],
         [-1.3875,  0.1807, -1.3536,  ..., -0.2598, -0.4838,  0.0272],
         [ 0.6085,  0.8426, -0.9788,  ...,  1.3599,  1.5634,  0.1699],
         [-0.2876,  0.2596,  1.3767,  ...,  0.3963,  1.2342,  0.7276]]],
       grad_fn=<UnsafeViewBackward0>)


###Layer Normalization

In [ ]:
torch.manual_seed(123)

# 5 dimensional example
# 2 rows with random values (determined by the seed)
batch_example = torch.rand(2, 5)
batch_example

tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740],
        [0.8665, 0.1366, 0.1025, 0.1841, 0.7264]])

In [ ]:
norm_layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = norm_layer(batch_example)
out

tensor([[0.0000, 0.0000, 0.4091, 0.6587, 0.3914, 0.0000],
        [0.0000, 0.0000, 0.1902, 0.3182, 0.6486, 0.0000]],
       grad_fn=<ReluBackward0>)

In [ ]:
# dim=0 applies batch normalization, not recommended for LLMs
# dim=-1 calculates the mean across the column of each row
# Layer normalization is independent of the sample size
# dim=-1 or dim=1, -1 is better since it's more general
mean = out.mean(dim=-1, keepdim=True)
mean

tensor([[0.2432],
        [0.1928]], grad_fn=<MeanBackward1>)

In [ ]:
var = out.var(dim=-1, keepdim=True) # Keeps the dimension of the input tensor
var

tensor([[0.0799],
        [0.0670]], grad_fn=<VarBackward0>)

In [ ]:
# Formula for layer norm
normed = ((out - mean) / torch.sqrt(var))
print(normed.var(dim=-1, keepdim=True))

tensor([[1.],
        [1.]], grad_fn=<VarBackward0>)


Layer normalization is actually a layer with learnable weights

####Layer Norm Class

Mean: $\mu=\frac{1}{H}\sum_{i=1}^{H}x_i$

Variance: $\sigma^2=\frac{1}{H}\sum_{i=1}^{H}(x_i-\mu)^2$

Normalization: $\hat{x}_i=\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}$

Scaling and shifting (Affine transformation): $y_i=\gamma_i\hat{x}_i+\beta_i$

Where:
- $x_i$ is the $i$-th element of the input vector.
- $H$ is the hidden size (the number of features or dimensions in the layer).
- $\epsilon$ is a small constant added to the variance for numerical stability (preventing division by zero).
- $\gamma$ and $\beta$ are learnable parameter vectors of the same dimension as the input, used to scale and shift the normalized values.

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super() .__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        # unbiased: True, the bigger the input the bigger
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        normed_x = ((x - mean) / torch.sqrt(var + self.eps))
        return self.scale * normed_x + self.shift

##Feedforward Network

GELU Approxximation

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super(GELU, self).__init__()

    def forward(self, x):
        return 0.5 * x * (
            1 + torch.tanh(
                torch.sqrt(torch.tensor(2 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))
            )
        )

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super(FeedForward, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
ffn = FeedForward(GPT_CONFIG_124M)
ffn(torch.rand(2, 3, 768))

tensor([[[-1.5938e-02,  1.3474e-02, -2.1874e-01,  ..., -1.4109e-02,
          -7.8105e-03, -5.7992e-02],
         [-3.0801e-02, -4.9926e-02, -2.8670e-01,  ..., -7.0210e-03,
           3.6155e-02, -3.9049e-02],
         [-1.2934e-01,  2.9413e-02, -3.7849e-01,  ..., -6.0954e-02,
          -4.8549e-02,  1.8631e-02]],

        [[ 9.9368e-03,  8.5113e-02, -2.3261e-01,  ...,  3.5924e-02,
           5.9007e-02,  3.3169e-04],
         [-1.3251e-02,  9.0142e-03, -2.7375e-01,  ...,  6.5309e-02,
           2.8315e-03, -3.0470e-02],
         [-1.8571e-02, -3.8353e-02, -2.3160e-01,  ..., -1.1986e-01,
          -2.5620e-02,  5.5219e-02]]], grad_fn=<ViewBackward0>)

##Adding Shortcut Connections

In [ ]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super(ExampleDeepNeuralNetwork, self).__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList(
            nn.Sequential(nn.Linear(layer_sizes[i], layer_sizes[i + 1]), GELU())
            for i in range(len(layer_sizes) - 1)
        )

    def forward(self, x):
        for layer in self.layers:
            layer_output = layer(x)
            # Add the input back to the output of the module
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer(x)
            # Regular NN
            else:
                x = layer(x)
        return x

def print_gradients(model, x):
    # Forward pass
    output = model(x)
    target = torch.tensor([0.])

    # Loss calculation
    loss = nn.MSELoss()(output, target)

    # Backward pass
    loss.backward()

    # Print gradients
    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"Mean gradient for {name}: {param.grad.abs().mean().item()}")

In [ ]:
# Size of 1 as output, we want to NN to predict that
layer_sizes = [3, 3, 3, 3, 3, 1]

In [ ]:
sample_input = torch.tensor([[1., 0., -1.]])

Try it out

In [ ]:
torch.manual_seed(123)
model_without_shortcuts = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=False)
print_gradients(model_without_shortcuts, sample_input)

Mean gradient for layers.0.0.weight: 0.00020173584925942123
Mean gradient for layers.1.0.weight: 0.00012011159560643137
Mean gradient for layers.2.0.weight: 0.0007152040489017963
Mean gradient for layers.3.0.weight: 0.0013988736318424344
Mean gradient for layers.4.0.weight: 0.005049645435065031


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [ ]:
model_with_shortcuts = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=True)
print_gradients(model_with_shortcuts, sample_input)

Mean gradient for layers.0.0.weight: 0.0014432291500270367
Mean gradient for layers.1.0.weight: 0.004846952389925718
Mean gradient for layers.2.0.weight: 0.004138893447816372
Mean gradient for layers.3.0.weight: 0.005915115587413311
Mean gradient for layers.4.0.weight: 0.032659437507390976


##Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(
            d_in = cfg["emb_dim"],
            d_out = cfg["emb_dim"],
            context_length = cfg["context_length"],
            num_heads = cfg["num_heads"],
            qkv_bias = cfg["qkv_bias"],
            dropout = cfg["drop_rate"]
        )
        self.ff = FeedForward(cfg)
        self.ln1 = LayerNorm(cfg["emb_dim"])
        self.ln2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # We copy the shortcut value to add it later
        shortcut = x
        x = self.ln1(x)
        # Shape is [batch, num_tokens, emb_size]
        x = self.att(x)
        x = self.drop_shortcut(x)
        # Add the original input through the shortcut connections
        x = shortcut + x

        # Same here
        shortcut = x
        x = self.ln2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = shortcut + x

        return x

In [ ]:
torch.manual_seed(123)
x = torch.rand(2, 4, 768)
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

In [ ]:
x.shape

torch.Size([2, 4, 768])

In [ ]:
output.shape

torch.Size([2, 4, 768])

##Coding GPT Model

In [ ]:
GPT_CONFIG_124M = {
    "emb_dim": 768,
    "num_layers": 12,
    "num_heads": 12,
    "context_length": 1024,
    "vocab_size": 50257,
    "drop_rate": 0.1,
    "qkv_bias": True,
}

In [ ]:
class GPTModel(nn.Module):
    # cfg = configuration, python dictionary
    def __init__(self, cfg):
        super(GPTModel, self).__init__()
        # From vocabulary size (50257 from the above cell) to embedding dimension (768)
        # From a 50257 dimensional vector to a 768 dimensional vector
        # From the the vocabulary size to an intermediate representation
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Repeated transformer blocks, by the amount of # of layers
        self.transf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["num_layers"])]
        )

        # Layer normalizations applies optimization steps to improve training
        # After normalization, the layers have zero-centered mean and unit variance
        self.final_norm = LayerNorm(cfg["emb_dim"])
        # From the embedding space to the vocabulary size
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        # [batch_size, sequence_length]
        # Create embeddings
        batch_size, seq_len = in_idx.shape
        tok_emb = self.tok_emb(in_idx)
        pos_emb = self.pos_emb(torch.arange(seq_len))

        # Pass x through all the layers
        # Dropout, transformer blocks, normalizations, take final logits
        # They represent a statistical distribution over the vocabulary
        x = tok_emb + pos_emb
        x = self.drop_emb(x)
        x = self.transf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits

Batch from earlier, "real" data

In [ ]:
batch

tensor([[   40,   588, 11875,   290],
        [   40,   588, 14256,   290]])

In [ ]:
torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)
out = model(batch)

In [ ]:
out.shape

torch.Size([2, 4, 50257])

####Counting Parameters

In [ ]:
batch.numel()

8

In [ ]:
total_parameters = sum(p.numel() for p in model.parameters())
print(f"{total_parameters:,}")
# 163M parameters

163,037,184


There is some weight sharing going on

##Generating Text

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):

        # First : in the slice, select all of the rows
        # -context_size: select up to the context_size, the remaining, (after the second :) is ignored
        # For example, if idx is [1, 2, 3, 4, 5] and context_size is 3, idx_cond would be [3, 4, 5].
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        # Extracting the last row of the logits
        # The model generates logits for the entire sequence
        # The slice then extract the logits for only the last token
        # Slicing operation like before
        # : select all batches
        # -1 take the last element along the sequence dimension (predicted token)
        # : after, select all dimensions
        logits = logits[:, -1, :]
        # From shape of [batch_size, seq_length, vocab_size]
        #   to [batch_size, vocab_size]

        prob = torch.softmax(logits, dim=-1)
        # argmax finds the index position with the highest value (highest probabilities)
        idx_next = torch.argmax(prob, dim=-1, keepdim=True)
        # concatenate the starting sequence with the predicted token
        # dim=1, sequence dimension
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [ ]:
start_context = "My cat is cute and"

encoded = tokenizer.encode(start_context)
print(f"Encoded: {encoded}")

Encoded: [3666, 3797, 318, 13779, 290]


In [ ]:
# Unseuqeeze adds an additional dimension
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print(f"Encoded tensor shape: {encoded_tensor.shape}")
print(f"Encoded tensor: {encoded_tensor}")

Encoded tensor shape: torch.Size([1, 5])
Encoded tensor: tensor([[ 3666,  3797,   318, 13779,   290]])


In [ ]:
out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=2,
    context_size=1024,
)

In [ ]:
out

tensor([[ 3666,  3797,   318, 13779,   290, 46297, 14100]])

In [ ]:
out = (out.squeeze(0)).tolist()

In [ ]:
print(tokenizer.decode(out))

My cat is cute and voltsmillion
